In [0]:
WITH ownership_counts AS (
    SELECT fpv.ownership_key, COUNT(fpv.car_num) AS car_amount
    FROM transport_lakehouse.gold.fact_private_vehicles fpv
    GROUP BY fpv.ownership_key

    UNION ALL

    SELECT fm.ownership_key, COUNT(fm.car_num) AS car_amount
    FROM transport_lakehouse.gold.fact_motorcycles fm
    GROUP BY fm.ownership_key
),
final_counts AS (
    SELECT
        o.ownership_key,
        o.ownership,
        SUM(oc.car_amount) AS total_car_amount
    FROM ownership_counts oc
    LEFT JOIN transport_lakehouse.gold.dim_ownership o
        ON o.ownership_key = oc.ownership_key
    GROUP BY o.ownership_key, o.ownership
)
SELECT
    ownership_key,
    ownership,
    ROUND(100.0 * total_car_amount / SUM(total_car_amount) OVER (), 2) AS pct
FROM final_counts
ORDER BY total_car_amount DESC